# 07 · Skills – J.A.R.V.I.S. learns a procedure 🧪

> 🧪 **Preview.** Skills and toolboxes are preview features: names and behaviour may change. This notebook uses `AIProjectClient(..., allow_preview=True)`.

A **tool** is something an agent can *call*. A **skill** is something an agent *knows how to do*: a folder with a `SKILL.md` (YAML front matter + Markdown instructions, the open [Agent Skills](https://agentskills.io) format). Skills keep rules like "how we escalate incidents" **out of every agent's prompt** and in one versioned place.

| 🚚 Delivery mode | 🧠 How the agent gets the skill | 🚦 Status in this repo |
|---|---|---|
| **Direct injection** | skill is downloaded into the agent project; the agent code adds it to its instructions | ✅ `jarvis-hosted` (chapter 09) |
| **Toolbox** | skill is attached to a toolbox; MCP clients read it as an MCP *resource* | ✅ shown below |
| **Prompt agent + GitHub Copilot harness** | skill reference in the agent definition | ℹ️ described, not run |

Skill versions are immutable; the skill tracks a `default_version`. Deep dive: [docs/mcp-a2a-skills.md](../docs/mcp-a2a-skills.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

import re
from pathlib import Path
from azure.ai.projects.models import SkillInlineContent

skills = project(preview=True).beta.skills
SKILL_FILE = ROOT / "skills" / "stark-escalation" / "SKILL.md"
text = SKILL_FILE.read_text(encoding="utf-8")
front, body = re.match(r"---\n(.*?)\n---\n(.*)", text, re.S).groups()
meta = dict(line.split(": ", 1) for line in front.splitlines())
print(meta["name"], "-", meta["description"])

## 1 · Author and upload (version 1)
`name` must be lowercase letters, digits and hyphens. Creating a version creates the skill if it does not exist yet.

In [ ]:
v1 = skills.create(name=meta["name"], inline_content=SkillInlineContent(description=meta["description"], instructions=body.strip()))
print("created", v1.name, "version", v1.version)

## 2 · A new version, then promote it
Edit the policy (here: critical incidents must also name a floor), create **version 2**, test, then make it the default – agents and toolboxes that follow the default pick it up **without any code change**.

In [ ]:
body2 = body.strip() + "\n\n## Addendum\n6. For critical incidents, also state which floor of the tower is affected.\n"
v2 = skills.create(name=meta["name"], inline_content=SkillInlineContent(description=meta["description"], instructions=body2))
print("created version", v2.version)

print("before:", skills.get(name=meta["name"]).default_version)
skills.update(meta["name"], default_version=v2.version)
print("after :", skills.get(name=meta["name"]).default_version)

for v in skills.list_versions(meta["name"]):
    print("  version", v.version)

## 3 · Download (direct injection)
The hosted agent (`src/jarvis-hosted`) loads every `skills/<name>/SKILL.md` into its instructions at startup. `scripts/sync_skills.py` copies the repo's `skills/` there on deploy; you can also pull a skill **from Foundry**:

In [ ]:
import io, zipfile
data = b"".join(skills.download_content(meta["name"]))
with zipfile.ZipFile(io.BytesIO(data)) as z:
    print(z.namelist())
    target = ROOT / "src" / "jarvis-hosted" / "skills" / meta["name"]
    target.mkdir(parents=True, exist_ok=True)
    for n in z.namelist():
        if n.endswith("SKILL.md"):
            (target / "SKILL.md").write_bytes(z.read(n))
print("saved to", target)

## 4 · Attach to a toolbox
A **toolbox** is a named, versioned bundle of tools (and skills) behind **one MCP endpoint**. Attach the skill; omit `version` to follow the default.

In [ ]:
from azure.ai.projects.models import ToolboxSkillReference
tb = project(preview=True).toolboxes.create_version(
    name="stark-toolbox",
    description="Escalation know-how for the workshop",
    tools=[],
    skills=[ToolboxSkillReference(name=meta["name"])],
)
print("toolbox version:", tb.version)
TOOLBOX_URL = f"{cfg('STARK_PROJECT_ENDPOINT')}/toolboxes/stark-toolbox/versions/{tb.version}/mcp?api-version=v1"
print(TOOLBOX_URL)

## 5 · Read the skill like any MCP client
Skills appear as **MCP resources** (`resources/list`, `resources/read`). Any MCP client can read them – no Foundry SDK needed. (Toolbox versions must be published/promoted to be served on the consumer endpoint; the version-specific URL above is for testing.)

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

token = credential().get_token("https://ai.azure.com/.default").token
headers = {"Authorization": f"Bearer {token}", "Foundry-Features": "Skills=V1Preview"}

async with streamablehttp_client(TOOLBOX_URL, headers=headers) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        resources = await session.list_resources()
        for res in resources.resources:
            print("resource:", res.uri, "-", res.name)

## 6 · Prompt agents + skills (GitHub Copilot harness, preview)
A prompt agent that uses the **GitHub Copilot harness** can reference a skill directly. In an `azure.yaml` it looks like this (from the Microsoft Learn docs; not run here):

```yaml
services:
  stark-skill:
    host: azure.ai.skill
    uses: [ai-project]
    archive: ./skills/stark-escalation
  my-agent:
    host: azure.ai.agent
    uses: [ai-project, stark-skill]
    kind: prompt
    harness:
      type: github_copilot_preview
    skills: [stark-skill]
```

## Clean up

In [ ]:
# keep the skill if you continue to chapter 09; uncomment to remove everything created here
# skills.delete(name=meta["name"])

## Recap
Skills move **procedures** out of prompts: author once, version, promote, deliver to many agents. Tools do things; skills teach how.

> 💡 Next: [08 · Agents from YAML](08_agents_from_yaml.ipynb).